# F1 Pit Stop Prediction - Exploratory Data Analysis

**Kaggle Playground Series S6E5**  
**Target:** PitNextLap (binary)  
**Metric:** AUC-ROC

This notebook covers everything we need to understand before touching a model. We go through data shape, class imbalance, feature distributions, tyre behaviour by compound, pit patterns across races, leakage checks, and train vs test consistency.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

PALETTE = {'no_pit': '#4C72B0', 'pit': '#DD8452'}
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f'Train: {train.shape[0]:,} rows, {train.shape[1]} columns')
print(f'Test:  {test.shape[0]:,} rows, {test.shape[1]} columns')

## 1. First Look

In [ ]:
train.head(10)

In [ ]:
train.dtypes

In [ ]:
train.describe().T

In [ ]:
# Missing values
missing_train = train.isnull().sum()
missing_test  = test.isnull().sum()
print('Missing in train:', missing_train[missing_train > 0].to_dict() or 'None')
print('Missing in test: ', missing_test[missing_test > 0].to_dict() or 'None')

## 2. Target Distribution and Class Imbalance

In [ ]:
target_counts = train['PitNextLap'].value_counts()
target_pct    = train['PitNextLap'].value_counts(normalize=True) * 100

print('Class counts:')
print(f'  No Pit (0): {target_counts[0]:,}  ({target_pct[0]:.1f}%)')
print(f'  Pit    (1): {target_counts[1]:,}  ({target_pct[1]:.1f}%)')
print(f'  Ratio: {target_counts[0]/target_counts[1]:.2f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['No Pit (0)', 'Pit (1)'], target_counts.values,
            color=[PALETTE['no_pit'], PALETTE['pit']], edgecolor='white', linewidth=1.2)
axes[0].set_title('Target class counts')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=['No Pit', 'Pit'],
            colors=[PALETTE['no_pit'], PALETTE['pit']],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Target class split')

plt.suptitle('Class Imbalance Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Tyre Life Analysis

TyreLife is likely one of the strongest signals. We want to understand what the tyre life looks like when a driver pits vs when they do not.

In [ ]:
compounds = train['Compound'].unique()
n = len(compounds)

fig, axes = plt.subplots(2, (n+1)//2, figsize=(16, 8))
axes = axes.flatten()

for i, compound in enumerate(sorted(compounds)):
    subset = train[train['Compound'] == compound]
    no_pit = subset[subset['PitNextLap'] == 0]['TyreLife']
    pit    = subset[subset['PitNextLap'] == 1]['TyreLife']

    axes[i].hist(no_pit, bins=30, alpha=0.6, color=PALETTE['no_pit'], label='No Pit', density=True)
    axes[i].hist(pit,    bins=30, alpha=0.6, color=PALETTE['pit'],    label='Pit',    density=True)
    axes[i].axvline(pit.median(), color=PALETTE['pit'], linestyle='--', linewidth=1.5,
                    label=f'Pit median: {pit.median():.0f}')
    axes[i].set_title(f'{compound}')
    axes[i].set_xlabel('TyreLife (laps)')
    axes[i].legend(fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('TyreLife Distribution by Compound (Pit vs No Pit)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('TyreLife at pit stop by compound:')
summary = train[train['PitNextLap'] == 1].groupby('Compound')['TyreLife'].agg(
    ['median', 'mean', 'std', 'min', 'max', 'count']
).round(1)
print(summary)

In [ ]:
# Pit probability as a function of TyreLife for each compound
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for compound in sorted(train['Compound'].unique()):
    subset = train[train['Compound'] == compound].copy()
    subset = subset[subset['TyreLife'] <= 60]
    pit_rate = subset.groupby('TyreLife')['PitNextLap'].mean()
    axes[0].plot(pit_rate.index, pit_rate.values, label=compound, linewidth=2)

axes[0].set_title('Pit probability vs TyreLife by compound')
axes[0].set_xlabel('TyreLife (laps)')
axes[0].set_ylabel('P(PitNextLap = 1)')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Box plot of TyreLife at pit by compound
pit_data = train[train['PitNextLap'] == 1]
compound_order = pit_data.groupby('Compound')['TyreLife'].median().sort_values().index
sns.boxplot(data=pit_data, x='Compound', y='TyreLife', order=compound_order,
            palette='Set2', ax=axes[1])
axes[1].set_title('TyreLife at pit stop by compound')
axes[1].set_xlabel('Compound')
axes[1].set_ylabel('TyreLife (laps)')

plt.tight_layout()
plt.show()

## 4. Race Progress and Pit Timing

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Pit rate across race progress bins
train['progress_bin'] = pd.cut(train['RaceProgress'], bins=25)
pit_by_progress = train.groupby('progress_bin', observed=True)['PitNextLap'].mean()

axes[0].bar(range(len(pit_by_progress)), pit_by_progress.values, color='steelblue', edgecolor='white')
axes[0].set_title('Pit probability across race progress')
axes[0].set_xlabel('Race progress (early to late)')
axes[0].set_ylabel('P(PitNextLap = 1)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_xticks([])
axes[0].annotate('Race start', xy=(0, 0), xytext=(1, pit_by_progress.values.max()*0.9),
                  fontsize=9, color='gray')
axes[0].annotate('Race end', xy=(len(pit_by_progress)-1, 0),
                  xytext=(len(pit_by_progress)-6, pit_by_progress.values.max()*0.9),
                  fontsize=9, color='gray')

# Distribution of RaceProgress for pit vs no pit
axes[1].hist(train[train['PitNextLap']==0]['RaceProgress'], bins=40, alpha=0.6,
             color=PALETTE['no_pit'], label='No Pit', density=True)
axes[1].hist(train[train['PitNextLap']==1]['RaceProgress'], bins=40, alpha=0.6,
             color=PALETTE['pit'], label='Pit', density=True)
axes[1].set_title('RaceProgress distribution: Pit vs No Pit')
axes[1].set_xlabel('Race Progress')
axes[1].set_ylabel('Density')
axes[1].legend()

train.drop(columns=['progress_bin'], inplace=True)
plt.tight_layout()
plt.show()

## 5. Lap Time and Degradation Signals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

num_features = ['LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'Position_Change']
titles = ['Lap Time (s)', 'Lap Time Delta', 'Cumulative Degradation', 'Position Change']

for ax, feat, title in zip(axes.flatten(), num_features, titles):
    no_pit_vals = train[train['PitNextLap']==0][feat].dropna()
    pit_vals    = train[train['PitNextLap']==1][feat].dropna()

    p1, p99 = no_pit_vals.quantile(0.01), no_pit_vals.quantile(0.99)
    no_pit_vals = no_pit_vals.clip(p1, p99)
    pit_vals    = pit_vals.clip(p1, p99)

    ax.hist(no_pit_vals, bins=50, alpha=0.6, color=PALETTE['no_pit'], label='No Pit', density=True)
    ax.hist(pit_vals,    bins=50, alpha=0.6, color=PALETTE['pit'],    label='Pit',    density=True)
    ax.axvline(no_pit_vals.median(), color=PALETTE['no_pit'], linestyle='--', linewidth=1.5)
    ax.axvline(pit_vals.median(),    color=PALETTE['pit'],    linestyle='--', linewidth=1.5)
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('Numeric Feature Distributions: Pit vs No Pit (dashed = median)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# KDE for Cumulative Degradation by compound
fig, ax = plt.subplots(figsize=(12, 5))

for compound in sorted(train['Compound'].unique()):
    subset = train[(train['Compound'] == compound) & (train['PitNextLap'] == 1)]
    if len(subset) > 50:
        subset['Cumulative_Degradation'].plot.kde(ax=ax, label=compound, linewidth=2)

ax.set_title('Cumulative Degradation at pit stop by compound')
ax.set_xlabel('Cumulative Degradation')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Pit Patterns by Driver, Compound and Circuit

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Pit rate by compound
compound_pit = train.groupby('Compound')['PitNextLap'].mean().sort_values(ascending=False)
axes[0].bar(compound_pit.index, compound_pit.values, color='steelblue', edgecolor='white')
axes[0].set_title('Pit rate by Compound')
axes[0].set_ylabel('P(PitNextLap = 1)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Pit rate by year
year_pit = train.groupby('Year')['PitNextLap'].mean()
axes[1].bar(year_pit.index.astype(str), year_pit.values, color='teal', edgecolor='white')
axes[1].set_title('Pit rate by Year')
axes[1].set_ylabel('P(PitNextLap = 1)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Top 20 circuits by pit rate
circuit_pit = train.groupby('Race')['PitNextLap'].mean().sort_values(ascending=False).head(20)
axes[2].barh(circuit_pit.index[::-1], circuit_pit.values[::-1], color='coral', edgecolor='white')
axes[2].set_title('Pit rate by Circuit (top 20)')
axes[2].set_xlabel('P(PitNextLap = 1)')
axes[2].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.suptitle('Pit Patterns Across Categories', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top and bottom drivers by pit rate (min 50 laps)
driver_pit = train.groupby('Driver').agg(
    pit_rate=('PitNextLap', 'mean'),
    laps=('PitNextLap', 'count')
).query('laps >= 50').sort_values('pit_rate')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Lowest pit rate
bottom15 = driver_pit.head(15)
axes[0].barh(bottom15.index, bottom15['pit_rate'], color='steelblue', edgecolor='white')
axes[0].set_title('Drivers with lowest pit rate (min 50 laps)')
axes[0].set_xlabel('Pit rate')
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Highest pit rate
top15 = driver_pit.tail(15)
axes[1].barh(top15.index, top15['pit_rate'], color='coral', edgecolor='white')
axes[1].set_title('Drivers with highest pit rate (min 50 laps)')
axes[1].set_xlabel('Pit rate')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.tight_layout()
plt.show()

## 7. Stint and Position Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Pit rate by stint number
stint_pit = train.groupby('Stint')['PitNextLap'].mean()
axes[0].bar(stint_pit.index, stint_pit.values, color='steelblue', edgecolor='white')
axes[0].set_title('Pit rate by Stint number')
axes[0].set_xlabel('Stint')
axes[0].set_ylabel('P(PitNextLap = 1)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

# Pit rate by position
pos_pit = train[train['Position'] <= 20].groupby('Position')['PitNextLap'].mean()
axes[1].plot(pos_pit.index, pos_pit.values, marker='o', color='coral', linewidth=2)
axes[1].set_title('Pit rate by Race Position')
axes[1].set_xlabel('Position (1 = leader)')
axes[1].set_ylabel('P(PitNextLap = 1)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

plt.tight_layout()
plt.show()

## 8. Feature Correlation with Target

In [ ]:
num_cols = ['LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)',
            'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress',
            'Position_Change', 'PitStop', 'Year']

corr = train[num_cols + ['PitNextLap']].corr()['PitNextLap'].drop('PitNextLap').sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['tomato' if x < 0 else 'steelblue' for x in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Pearson correlation with PitNextLap', fontsize=13, fontweight='bold')
ax.set_xlabel('Correlation coefficient')

for i, v in enumerate(corr.values):
    ax.text(v + (0.002 if v >= 0 else -0.002), i, f'{v:.3f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Full correlation heatmap
fig, ax = plt.subplots(figsize=(13, 10))
corr_matrix = train[num_cols + ['PitNextLap']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Feature correlation matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Leakage Check

PitStop tells us whether a pit happened on the current lap. A driver almost never pits two laps in a row, so PitStop=1 should strongly predict PitNextLap=0. This is a legitimate feature, not leakage, but worth confirming the relationship.

In [ ]:
crosstab = pd.crosstab(train['PitStop'], train['PitNextLap'], normalize='index') * 100
print('PitStop vs PitNextLap (row normalized %):')
print(crosstab.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
crosstab.plot(kind='bar', ax=ax, color=[PALETTE['no_pit'], PALETTE['pit']],
              edgecolor='white', rot=0)
ax.set_title('If pitted this lap, do they pit next lap?')
ax.set_xlabel('PitStop (this lap)')
ax.set_ylabel('Percentage')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(['No Pit Next', 'Pit Next'], title='PitNextLap')
plt.tight_layout()
plt.show()

## 10. Pit Stop Sequences - How Often Do Consecutive Pits Happen?

In [ ]:
# Sort to check consecutive pit patterns
sorted_train = train.sort_values(['Race', 'Year', 'Driver', 'LapNumber'])
sorted_train['prev_pit'] = sorted_train.groupby(['Race', 'Year', 'Driver'])['PitNextLap'].shift(1)

consec = sorted_train.dropna(subset=['prev_pit'])
consec_rate = consec.groupby('prev_pit')['PitNextLap'].mean()

print('If PitNextLap was 1 last lap, P(PitNextLap = 1 this lap):')
print(consec_rate.rename({0: 'Last lap: No Pit', 1: 'Last lap: Pit'}))

## 11. Heatmap - Compound x Circuit Pit Rate

In [ ]:
# Top 15 circuits by volume
top_circuits = train['Race'].value_counts().head(15).index
pivot = train[train['Race'].isin(top_circuits)].pivot_table(
    values='PitNextLap', index='Race', columns='Compound', aggfunc='mean'
) * 100

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5,
            ax=ax, cbar_kws={'label': 'Pit rate (%)'})
ax.set_title('Pit rate (%) by Circuit x Compound (top 15 circuits)', fontsize=13, fontweight='bold')
ax.set_xlabel('Compound')
ax.set_ylabel('Circuit')
plt.tight_layout()
plt.show()

## 12. Train vs Test Distribution Check

We need to make sure the test set looks like the training set. Large distribution shifts usually mean features will behave differently at inference.

In [ ]:
num_feats = ['LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)',
             'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change']

fig, axes = plt.subplots(3, 3, figsize=(18, 13))
axes = axes.flatten()

ks_results = []
for i, col in enumerate(num_feats):
    tr = train[col].dropna()
    te = test[col].dropna()

    axes[i].hist(tr, bins=40, alpha=0.5, label='Train', density=True,
                 color=PALETTE['no_pit'], edgecolor='none')
    axes[i].hist(te, bins=40, alpha=0.5, label='Test',  density=True,
                 color=PALETTE['pit'],    edgecolor='none')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

    ks_stat, ks_p = stats.ks_2samp(tr, te)
    ks_results.append({'Feature': col, 'KS Stat': round(ks_stat, 4), 'p-value': round(ks_p, 4)})

plt.suptitle('Train vs Test Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

ks_df = pd.DataFrame(ks_results).sort_values('KS Stat', ascending=False)
print('KS test results (high KS stat = distribution shift):')
print(ks_df.to_string(index=False))

In [ ]:
# Categorical overlap check
cat_cols = ['Compound', 'Year']
for col in cat_cols:
    train_vals = set(train[col].unique())
    test_vals  = set(test[col].unique())
    only_train = train_vals - test_vals
    only_test  = test_vals - train_vals
    print(f'{col}:')
    print(f'  Only in train: {only_train or "None"}')
    print(f'  Only in test:  {only_test or "None"}')

## 13. Key Takeaways

Fill this in after running all cells:

**Class imbalance:**
- 

**Strongest raw signals:**
- 

**Tyre life pit windows by compound:**
- SOFT: ~ laps
- MEDIUM: ~ laps
- HARD: ~ laps

**Race progress patterns:**
- 

**Leakage / data issues found:**
- 

**Train vs test distribution concerns:**
- 

**Features to engineer based on EDA:**
- 